# 13.7.词的相似性和类比任务

在 [13.4节](13.04_word2vec_pretraining.ipynb)中，我们在一个小的数据集上训练了一个word2vec模型，并使用它为一个输入词寻找语义相似的词。实际上，在大型语料库上预先训练的词向量可以应用于下游的自然语言处理任务，这将在后面的 第14章（自然语言处理：应用）中讨论。为了直观地演示大型语料库中预训练词向量的语义，让我们将预训练词向量应用到词的相似性和类比任务中。


## 13.7.1.环境配置

本节需要在昇腾 NPU 上运行 pypto 算子，因此需先完成 pypto 环境初始化。


In [2]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="On the interactive interface")
warnings.filterwarnings("ignore", message="Cannot create tensor")
import pypto
import torch
from torch import nn
from torch.nn import functional as F
import logging
logging.getLogger("torch_npu").setLevel(logging.WARNING)
import torch_npu
logging.getLogger("matplotlib").setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

In [3]:
import os
import torch

from src.utils import DATA_HUB, DATA_URL, download_extract
from src.pypto_ops import PyPTOMatmul

## 13.7.2.加载预训练词向量

以下列出维度为50、100和300的预训练GloVe嵌入，可从[GloVe网站](https://nlp.stanford.edu/projects/glove/)下载。预训练的fastText嵌入有多种语言。这里我们使用可以从[fastText网站](https://fasttext.cc/)下载300维度的英文版本（“wiki.en”）。


In [4]:
DATA_HUB['glove.6b.50d'] = (DATA_URL + 'glove.6B.50d.zip',
                           '0b8703943ccdb6eb788e6f091b8946e82231bc4d')

DATA_HUB['glove.6b.100d'] = (DATA_URL + 'glove.6B.100d.zip',
                            'cd43bfb07e44e6f27cbcc7bc9ae3d80284fdaf5a')

DATA_HUB['glove.42b.300d'] = (DATA_URL + 'glove.42B.300d.zip',
                             'b5116e234e9eb9076672cfeabf5469f3eec904fa')

DATA_HUB['wiki.en'] = (DATA_URL + 'wiki.en.zip',
                       'c1816da3821ae9f43899be655002f6c723e91b88')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
d2l.DATA_HUB[&#x27;glove.6b.50d&#x27;] = (d2l.DATA_URL + &#x27;glove.6B.50d.zip&#x27;,
                                &#x27;0b8703943ccdb6eb788e6f091b8946e82231bc4d&#x27;)
&nbsp;
d2l.DATA_HUB[&#x27;glove.6b.100d&#x27;] = (d2l.DATA_URL + &#x27;glove.6B.100d.zip&#x27;,
                                 &#x27;cd43bfb07e44e6f27cbcc7bc9ae3d80284fdaf5a&#x27;)
&nbsp;
d2l.DATA_HUB[&#x27;glove.42b.300d&#x27;] = (d2l.DATA_URL + &#x27;glove.42B.300d.zip&#x27;,
                                  &#x27;b5116e234e9eb9076672cfeabf5469f3eec904fa&#x27;)
&nbsp;
d2l.DATA_HUB[&#x27;wiki.en&#x27;] = (d2l.DATA_URL + &#x27;wiki.en.zip&#x27;,
                           &#x27;c1816da3821ae9f43899be655002f6c723e91b88&#x27;)
    </pre>
  </div>
</details>


为了加载这些预训练的GloVe和fastText嵌入，我们定义了以下`TokenEmbedding`类。


In [5]:
class TokenEmbedding:
    """GloVe嵌入"""
    def __init__(self, embedding_name):
        self.idx_to_token, self.idx_to_vec = self._load_embedding(
            embedding_name)
        self.unknown_idx = 0
        self.token_to_idx = {token: idx for idx, token in
                             enumerate(self.idx_to_token)}
        self.w_norm_sq = torch.sum(self.idx_to_vec * self.idx_to_vec,
                                   dim=1, keepdim=True)

    def _load_embedding(self, embedding_name):
        idx_to_token, idx_to_vec = ['<unk>'], []
        data_dir = download_extract(embedding_name)
        # GloVe网站：https://nlp.stanford.edu/projects/glove/
        # fastText网站：https://fasttext.cc/
        with open(os.path.join(data_dir, 'vec.txt'), 'r') as f:
            for line in f:
                elems = line.rstrip().split(' ')
                token, elems = elems[0], [float(elem) for elem in elems[1:]]
                # 跳过标题信息，例如fastText中的首行
                if len(elems) > 1:
                    idx_to_token.append(token)
                    idx_to_vec.append(elems)
        idx_to_vec = [[0] * len(idx_to_vec[0])] + idx_to_vec
        device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
        return idx_to_token, torch.tensor(idx_to_vec, device=device)

    def __getitem__(self, tokens):
        indices = [self.token_to_idx.get(token, self.unknown_idx)
                   for token in tokens]
        vecs = self.idx_to_vec[torch.tensor(indices)]
        return vecs

    def __len__(self):
        return len(self.idx_to_token)

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">行范数平方 <code>w_norm_sq</code> 一次性预计算（常驻 NPU），knn 每次查询直接复用，避免重复 O(词表大小×维度) 的计算与搬运。</li>
      <li style="margin: 0 0 8px 0;">词向量常驻 NPU：PyPTO kernel 要求输入在 NPU 上，且 40 万词表全量搬运代价高，加载时搬一次即可，knn 查询时零搬运。</li>
    </ul>
  </div>
</details>
<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
class TokenEmbedding:
    &quot;&quot;&quot;GloVe嵌入&quot;&quot;&quot;
    def __init__(self, embedding_name):
        self.idx_to_token, self.idx_to_vec = self._load_embedding(
            embedding_name)
        self.unknown_idx = 0
        self.token_to_idx = {token: idx for idx, token in
                             enumerate(self.idx_to_token)}
&nbsp;
    def _load_embedding(self, embedding_name):
        idx_to_token, idx_to_vec = [&#x27;&lt;unk&gt;&#x27;], []
        data_dir = d2l.download_extract(embedding_name)
        # GloVe网站：https://nlp.stanford.edu/projects/glove/
        # fastText网站：https://fasttext.cc/
        with open(os.path.join(data_dir, &#x27;vec.txt&#x27;), &#x27;r&#x27;) as f:
            for line in f:
                elems = line.rstrip().split(&#x27; &#x27;)
                token, elems = elems[0], [float(elem) for elem in elems[1:]]
                # 跳过标题信息，例如fastText中的首行
                if len(elems) &gt; 1:
                    idx_to_token.append(token)
                    idx_to_vec.append(elems)
        idx_to_vec = [[0] * len(idx_to_vec[0])] + idx_to_vec
        return idx_to_token, torch.tensor(idx_to_vec)
&nbsp;
    def __getitem__(self, tokens):
        indices = [self.token_to_idx.get(token, self.unknown_idx)
                   for token in tokens]
        vecs = self.idx_to_vec[torch.tensor(indices)]
        return vecs
&nbsp;
    def __len__(self):
        return len(self.idx_to_token)
    </pre>
  </div>
</details>


下面我们加载50维GloVe嵌入（在维基百科的子集上预训练）。创建`TokenEmbedding`实例时，如果尚未下载指定的嵌入文件，则必须下载该文件。


In [6]:
glove_6b50d = TokenEmbedding('glove.6b.50d')

正在从 https://d2l-data.s3-accelerate.amazonaws.com/glove.6B.50d.zip 下载 ../data/glove.6B.50d.zip...


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
glove_6b50d = TokenEmbedding(&#x27;glove.6b.50d&#x27;)
    </pre>
  </div>
</details>


输出词表大小。词表包含400000个词（词元）和一个特殊的未知词元。


In [7]:
len(glove_6b50d)

400001

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
len(glove_6b50d)
    </pre>
  </div>
</details>


我们可以得到词表中一个单词的索引，反之亦然。


In [8]:
glove_6b50d.token_to_idx['beautiful'], glove_6b50d.idx_to_token[3367]

(3367, 'beautiful')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
glove_6b50d.token_to_idx[&#x27;beautiful&#x27;], glove_6b50d.idx_to_token[3367]
    </pre>
  </div>
</details>


## 13.7.3.应用预训练词向量

使用加载的GloVe向量，我们将通过下面的词相似性和类比任务中来展示词向量的语义。

### 词相似度

与 [13.4.4节](13.04_word2vec_pretraining.ipynb)类似，为了根据词向量之间的余弦相似性为输入词查找语义相似的词，我们实现了以下`knn`（$k$近邻）函数。


### 13.7.3.1.cosine_sim 算子：余弦相似度融合 Kernel

matmul 算子已在[13.4节](13.04_word2vec_pretraining.ipynb)通过 `PyPTOMatmul` 使用（源自 [4.2 节](../04_pypto_multilayer_perceptrons/04.02_mlp_scratch.ipynb)），从 `src.pypto_ops` 直接导入即可复用。knn 的余弦相似度还需要一个新算子：**cosine_sim**（矩阵乘与归一化的融合）。

为保持教学完整性，**本节将内联展示该算子的完整 Kernel + 工厂函数链条**，帮助读者看清 knn 在 NPU 上的完整工作流。该算子同样收录在 `src/pypto_ops.py` 中，后续章节可直接导入。


In [9]:
@pypto.frontend.jit(runtime_options={"run_mode": pypto.RunMode.NPU})
def cosine_sim_fwd_kernel(
    w: pypto.Tensor([], pypto.DT_FP32),
    x: pypto.Tensor([], pypto.DT_FP32),
    w_norm_sq: pypto.Tensor([], pypto.DT_FP32),
    cos: pypto.Tensor([], pypto.DT_FP32),
    x_norm_sq: float,
):
    pypto.set_cube_tile_shapes([128, 128], [64, 128], [128, 128])
    pypto.set_vec_tile_shapes(128, 8)
    dot = pypto.matmul(w, x, pypto.DT_FP32)
    denom_sq = pypto.add(pypto.mul(w_norm_sq, x_norm_sq), 1e-9)
    cos.move(pypto.mul(dot, pypto.rsqrt(denom_sq)))

def make_pypto_cosine_sim(fwd_kernel):
    class PyPTOCosineSimImpl(torch.autograd.Function):
        @staticmethod
        def forward(ctx, w, x, w_norm_sq):
            cos = torch.empty(w.shape[0], 1, device=w.device, dtype=w.dtype)
            x_norm_sq = float(torch.sum(x * x))
            fwd_kernel(w, x, w_norm_sq, cos, x_norm_sq)
            return cos

        @staticmethod
        def backward(ctx, grad_cos):
            raise NotImplementedError(
                "PyPTOCosineSim 仅用于推理（knn 检索），不实现反向")
    return PyPTOCosineSimImpl

PyPTOCosineSim = make_pypto_cosine_sim(cosine_sim_fwd_kernel)


<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><b>cosine_sim 算子</b>：前向在单个 kernel 内完成 <code>W@x</code>（cube matmul）与 <code>rsqrt</code> 归一化（vec），避免矩阵乘与归一化之间的中间量落盘；<code>w_norm_sq</code>（行范数平方）在加载时预计算一次，<code>x_norm_sq</code> 是 d 维小量的一次归约，由宿主侧算好后以标量传入。</li>
      <li style="margin: 0 0 8px 0;">反向：该算子仅用于推理（knn 检索），不实现反向。</li>
      <li style="margin: 0 0 8px 0;">该算子已收录在 <code>src/pypto_ops.py</code> 中，后续章节可直接导入（如 <a href="./13.04_word2vec_pretraining.ipynb">13.4 节</a>的相似词查询）。</li>
      <li style="margin: 0 0 8px 0;"><b>kernel 接口</b>：输入 <code>w (V, d)</code>（词表×维度）、<code>x (d, 1)</code>（查询向量）、<code>w_norm_sq (V, 1)</code>（预计算的行范数平方）；输出 <code>(V, 1)</code> 余弦相似度。kernel 内：<code>dot = W@x</code>（cube），<code>cos = dot·rsqrt(w_norm_sq·x_norm_sq + 1e-9)</code>（vec）。</li>
    </ul>
  </div>
</details>


In [10]:
def knn(W, W_norm_sq, x, k):
    # 增加1e-9以获得数值稳定性
    x = x.reshape(-1, 1)
    cos = PyPTOCosineSim.apply(W, x, W_norm_sq).reshape(-1)
    topk_npu = torch.topk(cos, k=k)[1]
    return topk_npu.cpu(), cos[topk_npu].cpu()


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
def knn(W, x, k):
    # 增加1e-9以获得数值稳定性
    cos = torch.mv(W, x.reshape(-1,)) / (
        torch.sqrt(torch.sum(W * W, axis=1) + 1e-9) *
        torch.sqrt((x * x).sum()))
    _, topk = torch.topk(cos, k=k)
    return topk, [cos[int(i)] for i in topk]
    </pre>
  </div>
</details>


然后，我们使用`TokenEmbedding`的实例`embed`中预训练好的词向量来搜索相似的词。


In [11]:
def get_similar_tokens(query_token, k, embed):
    topk, cos = knn(embed.idx_to_vec, embed.w_norm_sq,
                    embed[[query_token]], k + 1)
    for i, c in zip(topk[1:], cos[1:]):  # 排除输入词
        print(f'{embed.idx_to_token[int(i)]}：cosine相似度={float(c):.3f}')

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
def get_similar_tokens(query_token, k, embed):
    topk, cos = knn(embed.idx_to_vec, embed[[query_token]], k + 1)
    for i, c in zip(topk[1:], cos[1:]):  # 排除输入词
        print(f&#x27;{embed.idx_to_token[int(i)]}：cosine相似度={float(c):.3f}&#x27;)
    </pre>
  </div>
</details>


`glove_6b50d`中预训练词向量的词表包含400000个词和一个特殊的未知词元。排除输入词和未知词元后，我们在词表中找到与“chip”一词语义最相似的三个词。


In [12]:
get_similar_tokens('chip', 3, glove_6b50d)

chips：cosine相似度=0.856
intel：cosine相似度=0.749
electronics：cosine相似度=0.749


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
get_similar_tokens(&#x27;chip&#x27;, 3, glove_6b50d)
    </pre>
  </div>
</details>


下面输出与“baby”和“beautiful”相似的词。


In [13]:
get_similar_tokens('baby', 3, glove_6b50d)

babies：cosine相似度=0.839
boy：cosine相似度=0.800
girl：cosine相似度=0.792


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
get_similar_tokens(&#x27;baby&#x27;, 3, glove_6b50d)
    </pre>
  </div>
</details>


In [14]:
get_similar_tokens('beautiful', 3, glove_6b50d)

lovely：cosine相似度=0.921
gorgeous：cosine相似度=0.893
wonderful：cosine相似度=0.830


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
get_similar_tokens(&#x27;beautiful&#x27;, 3, glove_6b50d)
    </pre>
  </div>
</details>


### 词类比

除了找到相似的词，我们还可以将词向量应用到词类比任务中。
例如，“man” : “woman” :: “son” : “daughter”是一个词的类比。
“man”是对“woman”的类比，“son”是对“daughter”的类比。
具体来说，词类比任务可以定义为：
对于单词类比$a : b :: c : d$，给出前三个词$a$、$b$和$c$，找到$d$。
用$\text{vec}(w)$表示词$w$的向量，
为了完成这个类比，我们将找到一个词，
其向量与$\text{vec}(c)+\text{vec}(b)-\text{vec}(a)$的结果最相似。


In [15]:
def get_analogy(token_a, token_b, token_c, embed):
    vecs = embed[[token_a, token_b, token_c]]
    x = vecs[1] - vecs[0] + vecs[2]
    topk, cos = knn(embed.idx_to_vec, embed.w_norm_sq, x, 1)
    return embed.idx_to_token[int(topk[0])]  # 删除未知词


<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
def get_analogy(token_a, token_b, token_c, embed):
    vecs = embed[[token_a, token_b, token_c]]
    x = vecs[1] - vecs[0] + vecs[2]
    topk, cos = knn(embed.idx_to_vec, x, 1)
    return embed.idx_to_token[int(topk[0])]  # 删除未知词
    </pre>
  </div>
</details>


让我们使用加载的词向量来验证“male-female”类比。


In [16]:
get_analogy('man', 'woman', 'son', glove_6b50d)

'daughter'

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
get_analogy(&#x27;man&#x27;, &#x27;woman&#x27;, &#x27;son&#x27;, glove_6b50d)
    </pre>
  </div>
</details>


下面完成一个“首都-国家”的类比：
“beijing” : “china” :: “tokyo” : “japan”。
这说明了预训练词向量中的语义。


In [17]:
get_analogy('beijing', 'china', 'tokyo', glove_6b50d)

'japan'

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
get_analogy(&#x27;beijing&#x27;, &#x27;china&#x27;, &#x27;tokyo&#x27;, glove_6b50d)
    </pre>
  </div>
</details>


另外，对于“bad” : “worst” :: “big” : “biggest”等“形容词-形容词最高级”的比喻，预训练词向量可以捕捉到句法信息。


In [18]:
get_analogy('bad', 'worst', 'big', glove_6b50d)

'biggest'

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
get_analogy(&#x27;bad&#x27;, &#x27;worst&#x27;, &#x27;big&#x27;, glove_6b50d)
    </pre>
  </div>
</details>


为了演示在预训练词向量中捕捉到的过去式概念，我们可以使用“现在式-过去式”的类比来测试句法：“do” : “did” :: “go” : “went”。


In [19]:
get_analogy('do', 'did', 'go', glove_6b50d)

'went'

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">
&nbsp;
get_analogy(&#x27;do&#x27;, &#x27;did&#x27;, &#x27;go&#x27;, glove_6b50d)
    </pre>
  </div>
</details>


## 13.7.4.小结

* 在实践中，在大型语料库上预先练的词向量可以应用于下游的自然语言处理任务。
* 预训练的词向量可以应用于词的相似性和类比任务。



## 13.7.5.练习

1. 使用`TokenEmbedding('wiki.en')`测试fastText结果。
1. 当词表非常大时，我们怎样才能更快地找到相似的词或完成一个词的类比呢？


参考答案详见 [answers/13.07_similarity_analogy_reference_answer](./answers/13.07_similarity_analogy_reference_answer.ipynb)。

### 13.7.5.1.参考答案（PyPTO）


In [20]:
!cat answers/txt/13.07_similarity_analogy_reference_answer_pypto.txt

### 13.7.5.2.参考答案（PyTorch）


In [21]:
!cat answers/txt/13.07_similarity_analogy_reference_answer_pytorch.txt